In [1]:

from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# Section 1

In [ ]:
def scaled_dot_product_attention(
    q: np.ndarray,
    k: np.ndarray,
    v: np.ndarray,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Pure-numpy attention.

    q: (..., T_q, d), k: (..., T_k, d), v: (..., T_k, d_v)
    mask: bool, broadcastable to (..., T_q, T_k), True = 允许注意。
    Returns (out, attn): (..., T_q, d_v) and (..., T_q, T_k).
    """
    attn = torch.einsum("...qd,...kd->...qk", q, k)
    attn /= math.sqrt(q.shape[-1])
    if mask is not None:
        attn = attn.masked_fill(~mask, -1e9)
    attn = torch.softmax(attn, dim=-1)
    out = torch.eisum('...qk,...kd -> ...qd', attn, v)
    return out, attn

def softmax(x, dim):
    x -= torch.max(x, dim=dim, keepdim=True)
    expx = torch.exp(x)
    return expx / expx.sum(dim=dim, keepdim=True)

def build_casual_mask(T):
    return torch.tril(torch.ones(T, T, dtype=torch.bool))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, heads, causal):
        self.heads = heads
        self.d_model = d_model
        self.d_head = d_model // heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.causal = causal
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.heads, self.d_head).transpose(1, 2)
        mask = build_casual_mask(T)
        out = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1,2).contiguous().view(B, T, self.d_model)
        return self.out_proj(out)

# Section 2

In [ ]:
def stable_softmax(x, axis):
    x = x.to(torch.float32)
    m = x - torch.max(x, dim=axis, keepdim=True)
    expm = torch.exp(m)
    return expm / expm.sum(axis=axis, keepdim=True)

def logsumexp(x, axis, keepdims):
    maxs = x.max(x, dim=axis, keepdim=True)
    out = maxs + torch.log(torch.sum(torch.exp(x - maxs), dim=axis, keepdim=True))
    if not keepdims:
        out = torch.squeeze(out, aixs=axis)
    return out

def log_softmax(x, axis):
    return x - logsumexp(x, axis, keepdims=True)

def cross_entropy(pred, labels):
    labels_idx = labels.unsqueeze(-1)
    logprob = log_softmax(pred, -1)
    target_logprob = torch.gather(logprob, -1, labels_idx).squeeze(-1)
    return torch.mean(-target_logprob)

def entropy_from_logits(logits, axis):
    logprob = log_softmax(logits, axis)
    return -torch.sum(torch.exp(logprob) * logprob)

def online_softmax_merge(
    state1: tuple[float, float], state2: tuple[float, float]
) -> tuple[float, float]:
    """合并两块的 (m, s) 状态; (m, s) 表示 logsumexp = m + log(s)。"""
    m1, s1 = state1
    m2, s2 = state2
    m = max(m1, m2)
    s = s * math.exp(m-m1) + s2 * math.exp(m-m2)
    return (m, s)


def streaming_entropy(chunks):
    m, s, t = -math.inf, 0, 0
    for chunk in chunks:
        c = np.array(chunk)
        if c.size == 0:
            continue
        cm = float(np.max(c))
        expc = np.exp(c-cm)
        cs = np.sum(expc)
        ct = np.sum(expc * c)
        new_m = max(m, cm)
        s = s * exp(m-new_m) + cs * exp(m-cm)
        t = t * np.exp(m-new_m) + cs * np.exp(m-new_m)
        m = new_m
    return m + np.log(s) - t / s

# Section 3

In [ ]:
def cross_entropy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # maxlogit = torch.max(logits, dim=-1, keepdim=True)
    # targets = targets.unsqueeze(-1)
    # explogit = torch.exp(logits - maxlogit)
    # explogitsum = torch.sum(explogit, dim=-1)
    # prob = torch.gather(explogit, -1, targets).unsqueeze(-1)
    # prob = prob-explogitsum
    # return - prob.mean()
    log_probs = F.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    return nll.mean()


def masked_cross_entropy(
    logits: torch.Tensor, targets: torch.Tensor, loss_mask: torch.Tensor
) -> torch.Tensor:
    mask = loss_mask.to(logits.dtype)
    safe_targets = torch.where(loss_mask.bool(), targets, torch.zeros_like(targets))
    log_probs = F.log_softmax(logits, dim=-1)
    nll = -log_probs.gather(-1, safe_targets.unsqueeze(-1)).squeeze(-1)
    return (nll * mask) / mask.sum().clamp(min=1.0)

def bce_with_logits(logits, targets):
    loss = logits.clamp(min=0) - logits * targets + torch.log1p(torch.exp(-logits.abs()))

def reward_model_pairwise_loss(
    chosen_scores: torch.Tensor, rejected_scores: torch.Tensor
) -> torch.Tensor:
    """Bradley-Terry pairwise loss: -log sigmoid(chosen - rejected), mean."""
    return -F.logsigmoid(chosen_scores - rejected_scores).mean()

def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
) -> torch.Tensor:
    """DPO loss: -log sigmoid(beta * (policy_logratio - ref_logratio)), mean.

    Inputs are per-sequence summed log-probs, shape (B,).
    """
    pi_logps = (policy_chosen_logps - policy_rejected_logps)
    ref_logps = ref_chosen_logps - ref_rejected_logps
    return -F.logsigmoid(beta * (pi_logps - ref_logps)).mean()

# Section 4

In [ ]:
def token_log_probs(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
    """log p(tokens[b, t]) under logits[b, t]. 调用方自己负责 shift 对齐."""
    logProb = F.log_softmax(logits, dim=-1)
    return logProb.gather(-1, tokens.unsqueeze(-1)).squeeze(-1)


def group_relative_advantage(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """GRPO: 组内标准化 (r - mean) / std; 组内全相同时返回全零 (信号为 0, 不该是 nan)."""
    rm = rewards.mean()
    return (rewards-rm) / torch.max(rewards.std(unbiased=False), eps)

def ppo_clipped_loss(
    logp_new: torch.Tensor,
    logp_old: torch.Tensor,
    adv: torch.Tensor,
    eps: float,
) -> torch.Tensor:
    unclipped = (logp_new - logp_old.detach()) * adv
    clipped = torch.clamp(logp_new - logp_old, 1 - eps, 1 + eps) * adv
    return -torch.min(unclipped, clipped).mean()

def kl_k3_estimator(logp_policy: torch.Tensor, logp_ref: torch.Tensor) -> torch.Tensor:
    """Schulman k3 估计 KL(policy || ref), 样本 x ~ policy.

    r = p_ref(x)/p_policy(x), k3 = r - log r - 1.
    性质: 逐点 >= 0 (因为 e^d >= d + 1), 且 E_policy[k3] 恰等于 KL (无偏).
    """
    return torch.exp(logp_ref - logp_policy) - (logp_ref - logp_policy) - 1

def grpo_token_loss(
    logp_new: torch.Tensor,       # [B, T]
    logp_old: torch.Tensor,       # [B, T]
    adv_per_seq: torch.Tensor,    # [B]  每条序列一个 advantage
    response_mask: torch.Tensor,  # [B, T]  1 = response token, 0 = prompt/padding
    kl: torch.Tensor,             # [B, T]  逐 token KL 估计
    beta: float,
    clip_eps: float = 0.2,
) -> torch.Tensor:
    adv = adv_per_seq.unsqueeze(-1)
    ratio = logp_new / logp_old
    clip = torch.clamp(ratio, 1-clip_eps, 1+clip_eps)
    per_token = -torch.min(clip * adv, ratio * adv) + beta*kl
    per_token *= response_mask.to(per_token.dtype)
    return per_token.sum() / response_mask.sum().clamp(min=1.0)
    


In [ ]:

def grpo(
    logits_new: torch.Tensor,     # [B, T, V] 当前 policy
    logits_old: torch.Tensor,     # [B, T, V] rollout 时的 policy
    logits_ref: torch.Tensor,     # [B, T, V] 冻结的 reference
    tokens: torch.Tensor,         # [B, T] prompt + response token ids
    response_mask: torch.Tensor,  # [B, T] 1 = response token
    rewards: torch.Tensor,        # [B] 每条序列一个标量奖励 (B == group size)
    clip_eps: float = 0.2,
    beta: float = 0.04,
) -> torch.Tensor:
    adv = group_relative_advantage(rewards)
    logp_new = token_log_probs(logits_new, tokens).detach()
    logp_old = token_log_probs(logits_new, tokens).detach()
    logp_ref = token_log_probs(logits_ref, tokens).detach()
    kl = kl_k3_estimator(logp_new, logp_ref)
    loss = grpo_token_loss(logp_new, logp_old, adv, response_mask, kl, beta, clip_eps)
    return loss

# Section 5 Sampling

In [ ]:
EPS = 1e-6
NEG_INF = float("-inf")

def apply_temperature(logits: torch.Tensor, T: float) -> torch.Tensor:
    """Scale logits by 1/T; T -> 0 collapses to a deterministic argmax distribution."""
    if T < EPS:
        out = torch.full_like(logits, NEG_INF)
        idx = logits.argmax(dim=-1, keepdim=True)
        out.scatter(-1, idx, 0)
        return out
    return logits/T

def top_k_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Keep the k largest logits per row, set the rest to -inf.

    k <= 0 or k >= vocab_size disables filtering. Ties at the k-th value are
    all kept (threshold semantics, same as HuggingFace).
    """
    if k <= 0 or k >= logits.shape[-1]:
        return logits.clone()
    kth_value = torch.topk(logits, k, dim=-1).values[..., -1, None]
    return logits.masked_fill(logits < kth_value, NEG_INF)

def top_p_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    """Nucleus filtering: keep the smallest prefix (in prob-descending order)
    whose cumulative probability reaches p; set the rest to -inf.

    p >= 1 disables filtering. The top-1 token is always kept.
    """
    if p >= 1:
        return logits.clone()
    sorted_logits, sorted_idx = torch.sort(logits, dim=-1, descending=True)
    cumprobs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
    sorted_remove = cumprobs >= p
    sorted_remove[..., 1:] = sorted_remove[..., :-1].clone()
    sorted_remove[..., 0] = False
    remove = torch.zero_like(sorted_remove).scatter(-1, sorted_idx, sorted_remove)
    return logits.masked_fill(remove, NEG_INF)


def sample_pipeline(
    logits: torch.Tensor,
    T: float,
    k: int,
    p: float,
    generator: Optional[torch.Generator] = None,
) -> torch.Tensor:
    """temperature -> top-k -> top-p -> softmax -> multinomial.

    Returns sampled token ids with shape logits.shape[:-1]
    (0-dim tensor for (V,) input, (B,) for (B, V) input).
    """

    logits = apply_temperature(logits, T)
    logits = top_k_filter(logits, k)
    logits = top_p_filter(logits, p)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1, generator=generator).squeeze(-1)


def greedy_decode(
    model_fn: Callable[[torch.Tensor], torch.Tensor],
    prompt_ids: torch.Tensor,
    max_new_tokens: int,
    stop_id: Optional[int] = None,
) -> torch.Tensor:
    """Autoregressive argmax decoding.

    model_fn: takes the full 1-D id sequence so far, returns next-token logits (V,).
    Returns prompt + generated ids; if stop_id is generated it is included and
    decoding stops early.
    """
    decoded = 0
    ids = prompt_ids.clone()
    while decoded < max_new_tokens:
        logits = model_fn(prompt_ids)
        next_id = logits.argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id])
        if stop_id is not None and next_id == stop_id:
            break
    return ids

# Section 6 Backward Autograd

In [ ]:

from __future__ import annotations

import math

import numpy as np

# ========================= COLD-WRITE ZONE =========================


def matmul_backward(
    A: np.ndarray, B: np.ndarray, dC: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """C = A @ B 的 backward。A:(m,k) B:(k,n) dC:(m,n) -> (dA, dB)."""
    da = dC @ B.T
    db = A.T @ dC
    return (da, db)


def linear_backward(
    x: np.ndarray, W: np.ndarray, b: np.ndarray, dY: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Y = x @ W + b 的 backward。x:(batch,in) W:(in,out) b:(out,)."""
    dX = dY @ W.T
    dW = x.T @ dY
    db = dY.sum(axis=0)
    return (dX, dW, db)
    